# Stage 1: Latent Demand Recovery (Hourly Imputation)
**Paper**: FreshRetailNet-50K (arXiv:2505.16319)  
**Method**: PyPOTS TimesNet on hourly sales data  
**Output**: `demand.parquet` with `sale_amount_pred` column  
**Estimated runtime**: 3-4h on Kaggle T4 GPU


## 0. Setup & Dependencies


In [1]:
import subprocess, sys, os, time, json, logging, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

T0 = time.time()
def elapsed_h(): return (time.time() - T0) / 3600

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s')
log = logging.getLogger(__name__)

# 1. Uninstall standard pypots to avoid import conflict
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "pypots"],
               capture_output=True)
for key in list(sys.modules.keys()):
    if 'pypots' in key:
        del sys.modules[key]

# 2. Install ONLY missing lightweight deps (torch is already on Kaggle, do NOT reinstall)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                       "einops", "pygrinder", "tsdb", "datasets", "pyarrow"])
# h5py, scipy, scikit-learn, sympy are pre-installed on Kaggle

# 3. Find custom PyPOTS in Kaggle input
log.info("Searching for custom PyPOTS...")
# List all input datasets for debugging
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.replace("/kaggle/input", "").count(os.sep)
    if depth <= 2:
        for d in dirs:
            log.info(f"  Found dir: {os.path.join(root, d)}")
        for f_name in files[:5]:
            log.info(f"  Found file: {os.path.join(root, f_name)}")
    if depth > 2:
        break

pypots_parent = None
# Search all possible locations
for root, dirs, files in os.walk("/kaggle/input"):
    if "pypots" in dirs:
        candidate = os.path.join(root, "pypots")
        if os.path.exists(os.path.join(candidate, "__init__.py")):
            pypots_parent = root
            break
    # Also check if __init__.py is directly in a pypots-named dir
    if os.path.basename(root) == "pypots" and "__init__.py" in files:
        pypots_parent = os.path.dirname(root)
        break

if pypots_parent:
    sys.path.insert(0, pypots_parent)
    log.info(f"Custom PyPOTS found at: {pypots_parent}/pypots/")
else:
    raise FileNotFoundError(
        "Custom PyPOTS not found in /kaggle/input/! "
        "Please add the 'custom-pypots' dataset as input to this notebook."
    )

# 4. Verify import
from pypots.imputation import TimesNet
from pypots.optim import Adam
import inspect
sig = inspect.signature(TimesNet.__init__)
has_OT = 'OT' in sig.parameters
log.info(f"TimesNet imported — OT parameter: {has_OT}")

from pathlib import Path
OUT = Path("/kaggle/working")
RESULTS = OUT / "results"
RESULTS.mkdir(exist_ok=True)

log.info(f"Setup done. Elapsed: {elapsed_h():.2f}h")

2026-05-23 09:20:46,288 Searching for custom PyPOTS...
2026-05-23 09:20:46,290   Found dir: /kaggle/input/datasets
2026-05-23 09:20:46,291   Found dir: /kaggle/input/datasets/phantrntngvyk64cntt
2026-05-23 09:20:46,291   Found dir: /kaggle/input/datasets/phantrntngvyk64cntt/custom-pypots
2026-05-23 09:20:46,301 Custom PyPOTS found at: /kaggle/input/datasets/phantrntngvyk64cntt/custom-pypots/pypots/
2026-05-23 09:20:53 [WARNING]: ‼️ PyPOTS Ecosystem configuration file does not exist.
2026-05-23 09:20:53 [INFO]: Wrote new configs to config.ini successfully.
2026-05-23 09:20:53 [INFO]: 💫 Initialized PyPOTS Ecosystem configuration file /root/.pypots/config.ini successfully.
2026-05-23 09:20:56.092056: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779528056.295084      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to registe

## 1. Load FreshRetailNet-50K


In [2]:
from datasets import load_dataset
import torch
import random

def set_seed(seed=1024):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(1024)

log.info("Loading FreshRetailNet-50K...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
data = ds['train'].to_pandas()
data = data.sort_values(by=['store_id', 'product_id', 'dt']).reset_index(drop=True)

# Also load eval for later use
if 'eval' in ds:
    eval_data = ds['eval'].to_pandas()
elif 'validation' in ds:
    eval_data = ds['validation'].to_pandas()
else:
    eval_data = None

log.info(f"Train: {len(data):,} rows, Eval: {len(eval_data) if eval_data is not None else 0:,} rows")
log.info(f"Columns: {list(data.columns)}")

# Verify hourly arrays
sample_hs = data['hours_sale'].iloc[0]
sample_hss = data['hours_stock_status'].iloc[0]
log.info(f"hours_sale type: {type(sample_hs)}, len: {len(sample_hs)}, first 6: {sample_hs[:6]}")
log.info(f"hours_stock_status type: {type(sample_hss)}, len: {len(sample_hss)}, first 6: {sample_hss[:6]}")
log.info(f"Elapsed: {elapsed_h():.2f}h")

2026-05-23 09:21:20,165 TensorFlow version 2.19.0 available.
2026-05-23 09:21:20,167 JAX version 0.7.2 available.
2026-05-23 09:21:21,609 Loading FreshRetailNet-50K...
2026-05-23 09:21:21,708 HTTP Request: HEAD https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-23 09:21:21,714 HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/Dingdong-Inc/FreshRetailNet-50K/08c1fab7f9257bc73679d415d65d644165d351d4/README.md "HTTP/1.1 200 OK"
2026-05-23 09:21:21,723 HTTP Request: GET https://huggingface.co/api/resolve-cache/datasets/Dingdong-Inc/FreshRetailNet-50K/08c1fab7f9257bc73679d415d65d644165d351d4/README.md "HTTP/1.1 200 OK"


README.md: 0.00B [00:00, ?B/s]

2026-05-23 09:21:21,758 HTTP Request: HEAD https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K/resolve/08c1fab7f9257bc73679d415d65d644165d351d4/FreshRetailNet-50K.py "HTTP/1.1 404 Not Found"
2026-05-23 09:21:21,797 HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/Dingdong-Inc/FreshRetailNet-50K/Dingdong-Inc/FreshRetailNet-50K.py "HTTP/1.1 404 Not Found"
2026-05-23 09:21:21,816 HTTP Request: GET https://huggingface.co/api/datasets/Dingdong-Inc/FreshRetailNet-50K/revision/08c1fab7f9257bc73679d415d65d644165d351d4 "HTTP/1.1 200 OK"
2026-05-23 09:21:21,893 HTTP Request: HEAD https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K/resolve/08c1fab7f9257bc73679d415d65d644165d351d4/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-05-23 09:21:21,951 HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=Dingdong-Inc/FreshRetailNet-50K "HTTP/1.1 200 OK"
2026-05-23 09:21:21,978 HTTP Request: GET https://huggingface.co/api/datas

data/train.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

2026-05-23 09:21:23,921 HTTP Request: HEAD https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K/resolve/08c1fab7f9257bc73679d415d65d644165d351d4/data/eval.parquet "HTTP/1.1 302 Found"


data/eval.parquet:   0%|          | 0.00/8.44M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

2026-05-23 09:21:38,173 Train: 4,500,000 rows, Eval: 350,000 rows
2026-05-23 09:21:38,175 Columns: ['city_id', 'store_id', 'management_group_id', 'first_category_id', 'second_category_id', 'third_category_id', 'product_id', 'dt', 'sale_amount', 'hours_sale', 'stock_hour6_22_cnt', 'hours_stock_status', 'discount', 'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']
2026-05-23 09:21:38,176 hours_sale type: <class 'numpy.ndarray'>, len: 24, first 6: [0. 0. 0. 0. 0. 0.]
2026-05-23 09:21:38,177 hours_stock_status type: <class 'numpy.ndarray'>, len: 24, first 6: [1 1 1 0 0 0]
2026-05-23 09:21:38,179 Elapsed: 0.02h


## 2. Prepare Hourly Arrays for Imputation
Following paper's `generate_data.py` exactly:
- Extract hours 6-22 (16 operating hours)
- Split 90 days into 3 windows of 30 days
- Mask stockout hours (status==1) as NaN
- Add covariates + hour position encoding


In [3]:
HORIZON = 90
OPERATING_HOURS_START = 6
OPERATING_HOURS_END = 22
N_HOURS = OPERATING_HOURS_END - OPERATING_HOURS_START  # 16
WINDOW_SIZE = 30  # days per window
N_WINDOWS = HORIZON // WINDOW_SIZE  # 3

series_num = len(data) // HORIZON  # 50000
log.info(f"Series: {series_num:,}, Windows per series: {N_WINDOWS}")

# 1. Extract hourly arrays
log.info("Extracting hourly arrays...")
hours_sale_raw = np.array(data['hours_sale'].tolist())          # (4500000, 24)
hours_stock_status_raw = np.array(data['hours_stock_status'].tolist())  # (4500000, 24)
log.info(f"hours_sale shape: {hours_sale_raw.shape}")
log.info(f"hours_stock_status shape: {hours_stock_status_raw.shape}")

# 2. Reshape to (series*windows, days_per_window, 24) and slice operating hours
hours_sale_full = hours_sale_raw.reshape(series_num * N_WINDOWS, WINDOW_SIZE, 24)
hours_stock_status_full = hours_stock_status_raw.reshape(series_num * N_WINDOWS, WINDOW_SIZE, 24)

# Keep only operating hours 6-22
hours_sale_origin = hours_sale_full[..., OPERATING_HOURS_START:OPERATING_HOURS_END]   # (150000, 30, 16)
hours_stock_status = hours_stock_status_full[..., OPERATING_HOURS_START:OPERATING_HOURS_END]  # (150000, 30, 16)

log.info(f"Operating hours shape: {hours_sale_origin.shape}")

# 3. Mask stockout hours as NaN (paper: status==1 means OOS)
hours_sale_masked = np.where(hours_stock_status == 1, np.nan, hours_sale_origin)
n_masked = np.isnan(hours_sale_masked).sum()
n_total = hours_sale_masked.size
log.info(f"Masked {n_masked:,}/{n_total:,} values ({n_masked/n_total*100:.1f}%) as NaN (stockout)")

# 4. Add covariates: discount, holiday_flag, precpt, avg_temperature
covariate_cols = ['discount', 'holiday_flag', 'precpt', 'avg_temperature']
covariate = data[covariate_cols].values.reshape(series_num * N_WINDOWS, WINDOW_SIZE, len(covariate_cols))
# Normalize covariates per-window (following paper)
covariate = covariate / (covariate.max(axis=1, keepdims=True) + 0.1)

# 5. Build feature tensor: [hourly_sale, covariates_broadcast, hour_position]
# hours_sale_masked: (150000, 30, 16) → (150000, 30, 16, 1)
# covariate: (150000, 30, 4) → broadcast to (150000, 30, 16, 4)
# hour_pos: arange(16)/15 → (1, 1, 16, 1) → broadcast

feat = np.concatenate([
    hours_sale_masked[..., None],  # (150000, 30, 16, 1)
    np.broadcast_to(covariate[:, :, None, :], hours_sale_masked.shape + (len(covariate_cols),)),  # (150000, 30, 16, 4)
    np.broadcast_to(np.arange(N_HOURS)[None, None, :, None] / (N_HOURS - 1),
                    hours_sale_masked[..., None].shape),  # (150000, 30, 16, 1)
], axis=-1)  # (150000, 30, 16, 6)

# Flatten time dimension: (150000, 30*16, 6) = (150000, 480, 6)
feat_flat = feat.reshape(-1, WINDOW_SIZE * N_HOURS, feat.shape[-1])
log.info(f"Feature tensor shape: {feat_flat.shape}")
log.info(f"NaN in feature[0] (hourly_sale): {np.isnan(feat_flat[:,:,0]).sum():,}")
log.info(f"Memory: {feat_flat.nbytes / 1e9:.2f} GB")
log.info(f"Elapsed: {elapsed_h():.2f}h")

2026-05-23 09:21:38,211 Series: 50,000, Windows per series: 3
2026-05-23 09:21:38,212 Extracting hourly arrays...
2026-05-23 09:21:41,407 hours_sale shape: (4500000, 24)
2026-05-23 09:21:41,408 hours_stock_status shape: (4500000, 24)
2026-05-23 09:21:41,410 Operating hours shape: (150000, 30, 16)
2026-05-23 09:21:42,018 Masked 14,311,536/72,000,000 values (19.9%) as NaN (stockout)
2026-05-23 09:21:43,186 Feature tensor shape: (150000, 480, 6)
2026-05-23 09:21:43,291 NaN in feature[0] (hourly_sale): 14,311,536
2026-05-23 09:21:43,292 Memory: 3.46 GB
2026-05-23 09:21:43,292 Elapsed: 0.02h


## 3. Train TimesNet Imputation Model
Using PyPOTS TimesNet to impute NaN values in stockout hours.
- n_steps = 480 (30 days × 16 hours)
- n_features = 6 (hourly_sale + 4 covariates + hour_position)
- epochs = 5, batch_size = 128 (following paper's CONFIG)


In [4]:
from pypots.imputation import TimesNet
from pypots.optim import Adam
import torch

DEVICE = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
log.info(f"Device: {DEVICE}")

# Check if custom fork supports OT
import inspect
sig = inspect.signature(TimesNet.__init__)
has_OT = 'OT' in sig.parameters

# Build model — following paper's CONFIG exactly
model_kwargs = dict(
    n_steps=WINDOW_SIZE * N_HOURS,     # 480
    n_features=feat_flat.shape[-1],     # 6
    n_layers=2,
    top_k=7,
    d_model=64,
    d_ffn=32,
    n_kernels=5,
    dropout=0.0,
    apply_nonstationary_norm=True,
    epochs=5,
    batch_size=128,
    patience=5,
    optimizer=Adam(lr=0.001, weight_decay=1e-5),
    device=DEVICE,
    saving_path=str(OUT / "timesnet_ckpt"),
)
if has_OT:
    model_kwargs['OT'] = 1  # Only impute first feature (hourly_sale)
    log.info("Using custom fork with OT=1")
else:
    log.info("Standard pypots — will apply explicit mask after imputation")

model = TimesNet(**model_kwargs)

train_set = {'X': feat_flat}
log.info(f"Training TimesNet on {feat_flat.shape[0]:,} samples, shape {feat_flat.shape}...")
log.info(f"This will take ~2-3 hours on GPU...")
model.fit(train_set)
log.info(f"Training done. Elapsed: {elapsed_h():.2f}h")

# Predict (impute missing values)
log.info("Running imputation prediction...")
results = model.predict(train_set)
imputation_raw = results['imputation']

# Handle CSDI-style 4D output
if len(imputation_raw.shape) == 4:
    imputation = imputation_raw.mean(axis=1)[:, :, :1]  # First feature only
else:
    imputation = imputation_raw[:, :, :1]  # First feature only (hourly_sale)

# Clip negative values to 0
imputation = np.where(imputation > 0, imputation, 0)
log.info(f"Imputation shape: {imputation.shape}")
log.info(f"Imputation stats: mean={imputation.mean():.4f}, std={imputation.std():.4f}, min={imputation.min():.4f}, max={imputation.max():.4f}")
log.info(f"Elapsed: {elapsed_h():.2f}h")

# Save raw imputation
np.save(OUT / "timesnet_imputation.npy", imputation)
log.info("Saved imputation to timesnet_imputation.npy")

# Free GPU memory
del model
torch.cuda.empty_cache()

2026-05-23 09:21:43,323 Device: cuda:0
2026-05-23 09:21:43,324 Using custom fork with OT=1
2026-05-23 09:21:43 [INFO]: Using the given device: cuda:0
2026-05-23 09:21:43 [INFO]: Model files will be saved to /kaggle/working/timesnet_ckpt/20260523_T092143
2026-05-23 09:21:43 [INFO]: Tensorboard file will be saved to /kaggle/working/timesnet_ckpt/20260523_T092143/tensorboard
2026-05-23 09:21:43 [INFO]: TimesNet initialized with the given hyperparameters, the number of trainable parameters: 1,354,438
2026-05-23 09:21:47,191 Training TimesNet on 150,000 samples, shape (150000, 480, 6)...
2026-05-23 09:21:47,192 This will take ~2-3 hours on GPU...
2026-05-23 09:53:17 [INFO]: Epoch 001 - training loss: 0.0626
2026-05-23 10:25:59 [INFO]: Epoch 002 - training loss: 0.0609
2026-05-23 10:58:47 [INFO]: Epoch 003 - training loss: 0.0605
2026-05-23 11:31:45 [INFO]: Epoch 004 - training loss: 0.0603
2026-05-23 12:07:24 [INFO]: Epoch 005 - training loss: 0.0602
2026-05-23 12:07:24 [INFO]: Finished tra

## 4. Demand Recovery
Replace stockout hours with imputed values, then sum to daily totals.
Formula: `d̃ = y ⊙ s + d̂ ⊙ (1-s)` where s=stock status mask


In [5]:
# Demand Recovery — following paper's app.py _demand_recovery()
# With custom fork (OT=1): imputation preserves observed values,
#   so we can directly replace hours 6-22 (paper's approach)
# Without OT: apply explicit mask to keep observed values

hours_sale_recovered = hours_sale_full.copy()  # (150000, 30, 24)
imputation_reshaped = imputation.squeeze(-1).reshape(-1, WINDOW_SIZE, N_HOURS)  # (150000, 30, 16)

if has_OT:
    # Paper's exact approach: replace all operating hours with imputation output
    # OT=1 ensures the model only modifies the target feature and preserves observed values
    hours_sale_recovered[..., OPERATING_HOURS_START:OPERATING_HOURS_END] = imputation_reshaped
    log.info("Recovery: used paper's direct replacement (OT=1 custom fork)")
else:
    # Explicit mask: only use imputed values where stockout, keep originals elsewhere
    stock_mask = hours_stock_status_full[..., OPERATING_HOURS_START:OPERATING_HOURS_END]
    hours_recovered_op = np.where(
        stock_mask == 1,           # stockout → use imputed
        imputation_reshaped,
        hours_sale_origin           # in-stock → keep original
    )
    hours_sale_recovered[..., OPERATING_HOURS_START:OPERATING_HOURS_END] = hours_recovered_op
    log.info("Recovery: used explicit mask (standard pypots fallback)")

# Sum 24 hours → daily recovered demand
sale_amount_pred = hours_sale_recovered.sum(axis=-1)  # (150000, 30)
sale_amount_pred = sale_amount_pred.reshape(-1, HORIZON)  # (50000, 90)

# Add to dataframe
data['sale_amount_pred'] = sale_amount_pred.reshape(-1)

# Sanity check
is_stockout = data['stock_hour6_22_cnt'] > 0
non_oos_diff = (data.loc[~is_stockout, 'sale_amount_pred'] - data.loc[~is_stockout, 'sale_amount']).abs().mean()
log.info(f"Sanity check — non-stockout days avg |diff|: {non_oos_diff:.6f} (should be ~0)")

log.info(f"Recovery stats:")
log.info(f"  sale_amount      — mean: {data['sale_amount'].mean():.4f}, std: {data['sale_amount'].std():.4f}")
log.info(f"  sale_amount_pred — mean: {data['sale_amount_pred'].mean():.4f}, std: {data['sale_amount_pred'].std():.4f}")
log.info(f"  Non-stockout: sale={data.loc[~is_stockout, 'sale_amount'].mean():.4f}, pred={data.loc[~is_stockout, 'sale_amount_pred'].mean():.4f}")
log.info(f"  Stockout:     sale={data.loc[is_stockout, 'sale_amount'].mean():.4f}, pred={data.loc[is_stockout, 'sale_amount_pred'].mean():.4f}")

recovery_changed = int((np.abs(data['sale_amount_pred'] - data['sale_amount']) > 1e-6).sum())
log.info(f"  Changed: {recovery_changed:,}/{len(data):,} rows ({recovery_changed/len(data)*100:.1f}%)")
log.info(f"Elapsed: {elapsed_h():.2f}h")

2026-05-23 12:27:20,035 Recovery: used paper's direct replacement (OT=1 custom fork)
2026-05-23 12:27:20,337 Sanity check — non-stockout days avg |diff|: 0.000000 (should be ~0)
2026-05-23 12:27:20,338 Recovery stats:
2026-05-23 12:27:20,365   sale_amount      — mean: 0.9986, std: 1.4067
2026-05-23 12:27:20,389   sale_amount_pred — mean: 1.2033, std: 1.6763
2026-05-23 12:27:20,507   Non-stockout: sale=0.9745, pred=0.9745
2026-05-23 12:27:20,592   Stockout:     sale=1.0289, pred=1.4913
2026-05-23 12:27:20,616   Changed: 1,914,546/4,500,000 rows (42.5%)
2026-05-23 12:27:20,618 Elapsed: 3.11h


## 5. Evaluation
Evaluate recovery quality using paper's decoupling score and bias metrics.


In [6]:
# === Paper's evaluation_decoupling (from app.py) ===
def evaluation_decoupling(df):
    """Compute decoupling score: correlation between stock_hours and demand
    should be LOWER for recovered demand (less censoring bias)."""
    df2 = df[['city_id', 'store_id', 'product_id', 'dt', 'holiday_flag', 'discount',
              'sale_amount', 'sale_amount_pred', 'stock_hour6_22_cnt']].copy()
    mu = (df2.query('stock_hour6_22_cnt==0')
          .groupby(['store_id', 'product_id'])['sale_amount']
          .mean().reset_index().rename(columns={'sale_amount': 'mu'}))
    corr = (df2.query('stock_hour6_22_cnt>0')
            .groupby(['store_id', 'product_id', 'holiday_flag'])
            .apply(lambda s: s[['stock_hour6_22_cnt', 'sale_amount', 'sale_amount_pred']].corr().iloc[:1, 1:])
            )
    stock_nunique = (df2.query('stock_hour6_22_cnt>0')
                     .groupby(['store_id', 'product_id', 'holiday_flag'])
                     .agg({'stock_hour6_22_cnt': 'nunique'}).reset_index()
                     .rename(columns={'stock_hour6_22_cnt': 'nunique'}))
    corr = (corr.reset_index()
            .merge(mu, on=['store_id', 'product_id'])
            .merge(stock_nunique.query('nunique>3'), on=['store_id', 'product_id', 'holiday_flag']))
    metric = pd.DataFrame({
        'method': ['sale_amount', 'sale_amount_pred'],
        'decoupling_score': np.nansum(
            corr[['sale_amount', 'sale_amount_pred']].values * corr[['mu']].values, axis=0
        ) / corr['mu'].sum()
    })
    return metric

try:
    decoup = evaluation_decoupling(data)
    print("\n=== Decoupling Score (lower = better, less censoring bias) ===")
    print(decoup.to_string(index=False))
except Exception as e:
    log.warning(f"Decoupling evaluation failed: {e}")

# === Recovery metrics ===
# On NON-stockout days: pred should match actual exactly (sanity check)
non_oos = data['stock_hour6_22_cnt'] == 0
wape_nonoos = np.sum(np.abs(data.loc[non_oos, 'sale_amount_pred'] - data.loc[non_oos, 'sale_amount'])) / np.sum(np.abs(data.loc[non_oos, 'sale_amount'])) * 100
wpe_nonoos = (data.loc[non_oos, 'sale_amount_pred'].sum() - data.loc[non_oos, 'sale_amount'].sum()) / data.loc[non_oos, 'sale_amount'].sum() * 100

# On ALL days: overall bias change
wpe_all = (data['sale_amount_pred'].sum() - data['sale_amount'].sum()) / data['sale_amount'].sum() * 100

print(f"\n=== Recovery Metrics ===")
print(f"  Non-stockout days WAPE: {wape_nonoos:.4f}% (should be ~0%)")
print(f"  Non-stockout days WPE:  {wpe_nonoos:.4f}% (should be ~0%)")
print(f"  Overall WPE (bias):     {wpe_all:.2f}% (positive = recovery added demand)")
print(f"  recovery_changed:       {recovery_changed:,}/{len(data):,} ({recovery_changed/len(data)*100:.1f}%)")

# Save metrics
metrics = {
    "wape_nonoos": float(wape_nonoos),
    "wpe_nonoos": float(wpe_nonoos),
    "wpe_all": float(wpe_all),
    "recovery_changed": int(recovery_changed),
    "recovery_changed_pct": float(recovery_changed / len(data) * 100),
}
json.dump(metrics, open(RESULTS / "task1_metrics.json", "w"), indent=2)
log.info(f"Elapsed: {elapsed_h():.2f}h")


=== Decoupling Score (lower = better, less censoring bias) ===
          method  decoupling_score
     sale_amount         -0.566447
sale_amount_pred          0.072971


2026-05-23 12:28:31,044 Elapsed: 3.13h



=== Recovery Metrics ===
  Non-stockout days WAPE: 0.0000% (should be ~0%)
  Non-stockout days WPE:  0.0000% (should be ~0%)
  Overall WPE (bias):     20.50% (positive = recovery added demand)
  recovery_changed:       1,914,546/4,500,000 (42.5%)


## 6. Save Checkpoint for Stage 2


In [7]:
# Save full recovered dataframe as checkpoint for Stage 2
CHECKPOINT = OUT / "demand.parquet"
data.to_parquet(CHECKPOINT, index=False)
log.info(f"Saved checkpoint: {CHECKPOINT} ({CHECKPOINT.stat().st_size / 1e6:.1f} MB)")

# Also save eval data for Stage 2
if eval_data is not None:
    eval_data.to_parquet(OUT / "eval_data.parquet", index=False)
    log.info(f"Saved eval data: {OUT / 'eval_data.parquet'}")

log.info("=" * 60)
log.info(f"STAGE 1 COMPLETE — Total time: {elapsed_h():.2f}h")
log.info("=" * 60)
log.info(f"Output files:")
for f in sorted(OUT.glob("*")):
    if f.is_file() and f.suffix in ['.parquet', '.npy', '.json']:
        log.info(f"  {f.name}: {f.stat().st_size / 1e6:.1f} MB")
log.info(f"\nNext: Add this notebook's output as dataset, then run Stage 2")

2026-05-23 12:28:46,185 Saved checkpoint: /kaggle/working/demand.parquet (81.4 MB)
2026-05-23 12:28:47,567 Saved eval data: /kaggle/working/eval_data.parquet
2026-05-23 12:28:47,568 ============================================================
2026-05-23 12:28:47,569 STAGE 1 COMPLETE — Total time: 3.13h
2026-05-23 12:28:47,570 ============================================================
2026-05-23 12:28:47,571 Output files:
2026-05-23 12:28:47,573   demand.parquet: 81.4 MB
2026-05-23 12:28:47,574   eval_data.parquet: 5.4 MB
2026-05-23 12:28:47,575   timesnet_imputation.npy: 288.0 MB
2026-05-23 12:28:47,575 
Next: Add this notebook's output as dataset, then run Stage 2
